# Few-Shot Learning Exploratory Analysis

This notebook demonstrates how to explore the Omniglot dataset and visualize experiment results.

In [ ]:
import sys
sys.path.append('../')

import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from src.data_loader import get_omniglot_dataset
from src.config import load_config

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

## Load Dataset

In [ ]:
config = load_config('../configs/base.yaml')
train_dataset, test_dataset = get_omniglot_dataset(config)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Number of classes in train: {len(train_dataset._dataset._alphabet_characters)}")
print(f"Number of classes in test: {len(test_dataset._dataset._alphabet_characters)}")

## Visualize Samples

In [ ]:
num_samples = 25
fig, axes = plt.subplots(5, 5, figsize=(10, 10))

for i in range(num_samples):
    img, label = train_dataset[i]
    row, col = i // 5, i % 5
    axes[row, col].imshow(img.squeeze(), cmap='gray')
    axes[row, col].set_title(f'Class: {label}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## Load and Visualize Experiment Results

In [ ]:
import os
import json

exp_id = '20240101_120000'
exp_dir = f'../results/{exp_id}'

if os.path.exists(exp_dir):
    metrics_df = pd.read_csv(os.path.join(exp_dir, 'metrics.csv'))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.lineplot(data=metrics_df, x='epoch', y='loss', ax=ax1, linewidth=2)
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    
    sns.lineplot(data=metrics_df, x='epoch', y='accuracy', ax=ax2, linewidth=2)
    ax2.set_title('Training Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    
    plt.tight_layout()
    plt.show()
    
    with open(os.path.join(exp_dir, 'metrics.json'), 'r') as f:
        final_metrics = json.load(f)
    print("Final Metrics:")
    print(final_metrics)
else:
    print(f"Experiment {exp_id} not found. Run an experiment first.")

## Compare Multiple Experiments

In [ ]:
results_dir = '../results'
exp_ids = []

for dir_name in os.listdir(results_dir):
    if os.path.isdir(os.path.join(results_dir, dir_name)):
        exp_ids.append(dir_name)

print(f"Found {len(exp_ids)} experiments")

results = []
for exp_id in exp_ids:
    exp_dir = os.path.join(results_dir, exp_id)
    config_path = os.path.join(exp_dir, 'config_used.yaml')
    metrics_path = os.path.join(exp_dir, 'metrics.json')
    
    if os.path.exists(config_path) and os.path.exists(metrics_path):
        import yaml
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        with open(metrics_path, 'r') as f:
            metrics = json.load(f)
        
        results.append({
            'exp_id': exp_id,
            'name': config['experiment']['name'],
            'model': config['model']['type'],
            'ways': config['data']['test_ways'],
            'shots': config['data']['test_shots'],
            'accuracy': metrics.get('test_accuracy', {}).get('mean', metrics.get('test_accuracy'))
        })

if results:
    df = pd.DataFrame(results)
    print(df)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=df, x='name', y='accuracy', ax=ax)
    ax.set_title('Experiment Comparison')
    ax.set_xlabel('Experiment')
    ax.set_ylabel('Accuracy')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()